# Kannolo Sparse Multivector Rerank Index Benchmark (LoTTE)

Build and search a SparseMultivecRerankIndex: first stage retrieval with HNSW on sparse vectors followed by half-precision (f16) multivector reranking. 

For compressed multivector reranking look at SparseMultivecTwoLevelsPQRerankIndex at pylib/mod.rs

## Import Required Libraries

In [ ]:
from kannolo import SparseMultivecRerankIndex
import ir_datasets
import ir_measures
import numpy as np
import pandas as pd
import time
from pathlib import Path
import os

## Load Dataset and Relevance Judgments

In [ ]:
# Load qrels from TSV file
qrels_path = ""  
qrels_df = pd.read_csv(qrels_path, sep="\t", header=None, names=["query_id", "Q0", "doc_id", "relevance"])
print(f"Loaded {len(qrels_df)} qrels from {qrels_path}")

# Convert to ir_measures QRel format - map query indices to actual query IDs
queries_ids_path = ""
queries_ids = np.load(queries_ids_path, allow_pickle=True)
print(f"Loaded {len(queries_ids)} query IDs from {queries_ids_path}")

# Map query indices in qrels to actual query IDs
qrels_df['query_id'] = qrels_df['query_id'].apply(lambda x: str(queries_ids[x]))
qrels_df['doc_id'] = qrels_df['doc_id'].astype(str)

qrels = [ir_measures.Qrel(str(row['query_id']), str(row['doc_id']), int(row['relevance'])) 
         for _, row in qrels_df.iterrows()]

# Get number of queries from queries_ids array
n_queries = len(queries_ids)
print(f"Total queries available: {n_queries}")

## Load Pre-built Index and Data

In [ ]:
# Path to pre-built sparse first-stage HNSW index
sparse_index_path = ""  

# Path to multivector data folder (containing documents.npy, doclens.npy, queries.npy)
multivec_data_folder = ""  

# Path to sparse queries folder (components.npy, values.npy, offsets.npy)
# If sparse queries are in seisic's bin format, use scripts/convert_bin_to_npy_arrays.py to convert them to numpy arrays
sparse_queries_path = ""

# Path to document ID mapping (maps index to actual doc_id)
doc_ids_path = "" 

print(f"Loading sparse index from {sparse_index_path}")
print(f"Loading multivec data from {multivec_data_folder}")
print(f"Loading sparse queries from {sparse_queries_path}")
print(f"Loading doc_ids from {doc_ids_path}")

In [ ]:
# Load sparse queries
query_components = np.load(Path(sparse_queries_path) / "components.npy").astype(np.int32)
query_values = np.load(Path(sparse_queries_path) / "values.npy").astype(np.float32)
query_offsets = np.load(Path(sparse_queries_path) / "offsets.npy").astype(np.int64)
n_sparse_queries = len(query_offsets) - 1
print(f"Loaded sparse queries: {n_sparse_queries} queries")

# Load dense multivector queries
multivec_queries = np.load(Path(multivec_data_folder) / "queries.npy").astype(np.float32)
print(f"Loaded multivec queries shape: {multivec_queries.shape}")
n_tokens = multivec_queries.shape[1]
token_dim = multivec_queries.shape[2]
print(f"Tokens per query: {n_tokens}, Token dimension: {token_dim}")

# Flatten multivector queries for batch processing
multivec_queries_flat = multivec_queries.reshape(-1).astype(np.float32)
print(f"Flattened multivec queries size: {len(multivec_queries_flat)}")

## Build Rerank Index

In [ ]:
# Build rerank index from pre-built sparse index and multivector data
start = time.time()
rerank_index = SparseMultivecRerankIndex.build_from_file(sparse_index_path, multivec_data_folder)
end = time.time()
print(f"Rerank index loaded in {end - start:.2f} seconds")

# Load document ID mapping (converts result indices to actual doc_ids)
doc_ids_map = np.load(doc_ids_path, allow_pickle=True)
print(f"Loaded doc_ids mapping with {len(doc_ids_map)} documents")

## Search with Multiple Parameter Combinations

In [ ]:
# Parameter configurations to test
configs = [
    {"k_candidates": 25, "ef_search": 50, "alpha": 0.025},
    {"k_candidates": 100, "ef_search": 100, "alpha": 0.025},
    {"k_candidates": 200, "ef_search": 200, "alpha": 0.1}
]

k = 10  # final top-k results
results = []

for config in configs:
    k_candidates = config["k_candidates"]
    ef_search = config["ef_search"]
    alpha = config["alpha"]
    
    # Search
    start = time.time()
    dists, ids = rerank_index.search(
        query_components=query_components,
        query_values=query_values,
        sparse_offsets=query_offsets,
        multivec_queries=multivec_queries_flat,
        n_tokens=n_tokens,
        token_dim=token_dim,
        k_candidates=k_candidates,
        k=k,
        ef_search=ef_search,
        alpha=alpha
    )
    elapsed = time.time() - start
    
    # Build results as DataFrame for ir_measures
    # Map query indices and document indices to actual IDs
    result_rows = []
    result_idx = 0
    
    for q_idx in range(n_sparse_queries):
        # Map query index to actual query ID
        actual_query_id = str(queries_ids[q_idx])
        
        for rank in range(k):
            if result_idx < len(ids):
                # Map the document index to actual doc_id using doc_ids_map
                doc_index = int(ids[result_idx])
                actual_doc_id = str(doc_ids_map[doc_index])
                score = float(dists[result_idx])
                result_rows.append({
                    'query_id': actual_query_id,
                    'doc_id': actual_doc_id,
                    'rank': rank,
                    'score': score
                })
                result_idx += 1
    
    res_df = pd.DataFrame(result_rows)
    
    # Ensure all IDs are strings for ir_measures
    res_df['query_id'] = res_df['query_id'].astype(str)
    res_df['doc_id'] = res_df['doc_id'].astype(str)
    
    # Evaluate with Success@5 metric using DataFrame
    metric = ir_measures.parse_measure("Success@5")
    metric_value = ir_measures.calc_aggregate([metric], qrels_df, res_df)[metric]
    avg_time = (elapsed / n_sparse_queries) * 10**6  # us per query
    
    results.append({
        "k_candidates": k_candidates,
        "ef_search": ef_search,
        "alpha": alpha,
        "avg_time_us": avg_time,
        "success_at_5": metric_value
    })
    
    print(f"k_cand={k_candidates}, ef={ef_search}, alpha={alpha}: "
          f"avg_time={avg_time:.3f}us, Success@5={metric_value:.3f}")

print("\n" + "="*60)
for r in results:
    print(f"Config: k_candidates={r['k_candidates']}, ef_search={r['ef_search']}, alpha={r['alpha']}")
    print(f"  Avg Query Time: {r['avg_time_us']:.3f} us")
    print(f"  Success@5: {r['success_at_5']:.3f}")
    print()